# Designing the diatonic neighborhood function

I am designing a function called `diatonic_neighborhood_around_pitch_set`. 

Here's the description:

```
Find the set of pitches (belonging to a pitch class set) in
the neighborhood (of a given proximity in semitones) of the pitches
in a pitch set.

Useful for finding successive chord voicings that have smooth voice
leading with the previous chord.
```

For each pitch in the pitch set, we take its neighborhood. So if the first pitch is 4 and the proximity is 3, then the neighborhood is 1 to 7. Then we keep the ones that are in the pitch class set. Add all of them to our final pitch set, ignore duplicates. That's it!

So I've added `harmonica.pitch.diatonic_neighborhood_around_pitch_set`, and it does exactly this.

## We're not there yet

This provides us with the set of pitches we can select. But now we need something that returns a list of pitch sets.

How do I do this? Like, what if the chord is [0,3,7], and the pcset is [0,5,9,10] and the proximity is 3.


In [ ]:
from harmonica import pitch

pset = pitch.PitchSet([0, 3, 7])
pcset = pitch.PitchClassSet([0, 5, 9, 10], 12)
neighborhood = pitch.diatonic_neighborhood(pset, pcset, 3)

print(neighborhood)

PitchSet(pitches=[-3, -2, 0, 5, 9, 10])


Oh, wait. This isn't how I want `pitch.diatonic_neighborhood_around_pitch_set` to work. Instead of making a flat list of pitches, it should create a list of lists of pitches that are proximal to each chord. Then you take the cartesian product of those lists and you pack each of the lists into a PitchSet object.

Let me fix that...

Okay, great! I've also renamed the function `diatonic_neighborhood`, because I think that's descriptive enough.

So now that we have a function that returns the diatonic neighborhood of a pitch set and a pitch class set, I can try my hand at another progression building algorithm.

Previously, we had this builder:

In [3]:
import random

from harmonica.pitch._changes import PCSetSeq
from harmonica.pitch._melody import PCSequence
from harmonica.pitch._scales import PitchClassSet


def ez_progression_builder(scale: PitchClassSet, pcseq: PCSequence) -> PCSetSeq:
    """Builds a chord progression (in the form of a sequence of pitch class sets)
    where the chords all belong to a particular scale, and the progression
    embeds a sequence of pitch classes (which can represent something like a bassline.)
    """

    progression: list[PitchClassSet] = []

    # For each pitch class in the sequence, choose 2 other pitch classes
    # from the scale at random.

    for pitch_class in pcseq.pitch_classes:
        # These are the other pitch classes to choose from
        other_pcs: list[int] = [pc for pc in scale.pitch_classes if pc != pitch_class]
        # Sample two of them at random
        sample = random.sample(other_pcs, 2)
        # Combine with pitch class from sequence to yield the chord
        chord_pcs = sorted([pitch_class] + sample)
        next_chord = PitchClassSet(chord_pcs, 12, pitch_class)
        progression.append(next_chord)

    return PCSetSeq(progression)


pcset = PitchClassSet([0, 2, 4, 5, 7, 9, 11], 12, 0)
pcseq = PCSequence([0, 11, 7, 4, 5], 12)
prog = ez_progression_builder(pcset, pcseq)
for pcset in prog:
    print(pcset)

PitchClassSet(pitch_classes=[0, 4, 9], modulus=12, root=0)
PitchClassSet(pitch_classes=[0, 4, 11], modulus=12, root=11)
PitchClassSet(pitch_classes=[5, 7, 9], modulus=12, root=7)
PitchClassSet(pitch_classes=[2, 4, 7], modulus=12, root=4)
PitchClassSet(pitch_classes=[2, 4, 5], modulus=12, root=5)


This yields a sequence of pitch class sets for us to generate chords from. 

We're not ready to start printing notes yet, though. 